# Prétraitement des textes

Dans cette partie, nous préparons les avis Amazon avant l'entraînement des modèles de classification.  
L'objectif est de nettoyer les textes, de préparer les labels et de créer un jeu de données exploitable pour les modèles.

In [2]:
import pandas as pd
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)

## Chargement du dataset

Nous commençons par charger le dataset utilisé dans l'étape d'exploration.

In [3]:
file_path = "../data/amazon_reviews.txt"
df = pd.read_csv(file_path, sep="\t")
df.head()

,DOC_ID,LABEL,RATING,VERIFIED_PURCHASE,PRODUCT_CATEGORY,PRODUCT_ID,PRODUCT_TITLE,REVIEW_TITLE,REVIEW_TEXT
0,1,__label1__,4,N,PC,B00008NG7N,"Targus PAUK10U Ultra Mini USB Keypad, Black",useful,"When least you think so, this product will save the day. Just keep it around just in case you need it for something."
1,2,__label1__,4,Y,Wireless,B00LH0Y3NM,Note 3 Battery : Stalion Strength Replacement 3200mAh Li-Ion Battery for Samsung Galaxy Note 3 [24-Month Warranty] with NFC Chip + Google Wallet Capable,New era for batteries,Lithium batteries are something new introduced in the market there average developing cost is relatively high but Stallion doesn't compromise on quality and provides us with the best at a low cost...
2,3,__label1__,3,N,Baby,B000I5UZ1Q,"Fisher-Price Papasan Cradle Swing, Starlight",doesn't swing very well.,I purchased this swing for my baby. She is 6 months now and has pretty much out grown it. It is very loud and doesn't swing very well. It is beautiful though. I love the colors and it has a lot of...
3,4,__label1__,4,N,Office Products,B003822IRA,Casio MS-80B Standard Function Desktop Calculator,Great computing!,I was looking for an inexpensive desk calcolatur and here it is. It works and does everything I need. Only issue is that it tilts slightly to one side so when I hit any keys it rocks a little bit....
4,5,__label1__,4,N,Beauty,B00PWSAXAM,Shine Whitening - Zero Peroxide Teeth Whitening System - No Sensitivity,Only use twice a week,"I only use it twice a week and the results are great. I have used other teeth whitening solutions and most of them, for the same results I would have to use it at least three times a week. Will ke..."


## Sélection des colonnes utiles

Pour la tâche de classification, nous conservons le texte de l’avis, la classe associée et le statut d’achat vérifié. Nous gardons également `DOC_ID` afin de pouvoir identifier chaque avis pendant les étapes de vérification et de prétraitement.

La longueur de chaque avis est calculée à partir de la colonne `REVIEW_TEXT` en comptant le nombre de mots.

In [4]:
df_text = df[
    [
        "DOC_ID",
        "REVIEW_TEXT",
        "VERIFIED_PURCHASE",
        "LABEL"
    ]
].copy()

df_text["review_word_count"] = (
    df_text["REVIEW_TEXT"]
    .astype(str)
    .str.split()
    .str.len()
)

df_text.head()

,DOC_ID,REVIEW_TEXT,VERIFIED_PURCHASE,LABEL,review_word_count
0,1,"When least you think so, this product will save the day. Just keep it around just in case you need it for something.",N,__label1__,23
1,2,Lithium batteries are something new introduced in the market there average developing cost is relatively high but Stallion doesn't compromise on quality and provides us with the best at a low cost...,Y,__label1__,69
2,3,I purchased this swing for my baby. She is 6 months now and has pretty much out grown it. It is very loud and doesn't swing very well. It is beautiful though. I love the colors and it has a lot of...,N,__label1__,52
3,4,I was looking for an inexpensive desk calcolatur and here it is. It works and does everything I need. Only issue is that it tilts slightly to one side so when I hit any keys it rocks a little bit....,N,__label1__,44
4,5,"I only use it twice a week and the results are great. I have used other teeth whitening solutions and most of them, for the same results I would have to use it at least three times a week. Will ke...",N,__label1__,66


## Renommage des labels

Nous remplaçons les labels d’origine par des noms plus explicites afin de faciliter la lecture et les étapes suivantes.

In [5]:
df_text["LABEL"] = df_text["LABEL"].replace({
    "__label1__": "fake",
    "__label2__": "real"
})

df_text["LABEL"].value_counts()

LABEL
fake    10500
real    10500
Name: count, dtype: int64

## Encodage des labels

Pour l’entraînement des modèles de classification, nous créons une version numérique de la variable cible.

Dans notre cas, la classe positive correspond aux faux avis :

- `fake` : 1
- `real` : 0

In [6]:
df_text["label_num"] = df_text["LABEL"].map({
    "fake": 1,
    "real": 0
})

df_text[["DOC_ID", "LABEL", "label_num"]].head()

,DOC_ID,LABEL,label_num
0,1,fake,1
1,2,fake,1
2,3,fake,1
3,4,fake,1
4,5,fake,1


## Prétraitement des textes

Nous appliquons un prétraitement sur les avis afin de réduire le bruit et d'obtenir des textes plus homogènes pour les modèles de classification.

Les opérations réalisées sont :

- conversion du texte en minuscules ;
- suppression de la ponctuation et des caractères spéciaux ;
- suppression des espaces multiples ;
- tokenisation du texte en mots ;
- suppression des mots vides ;
- racination des mots.

Le filtrage des termes très rares ou très fréquents sera réalisé plus tard lors de la représentation du texte avec les vectoriseurs `CountVectorizer` et `TfidfVectorizer`.

In [11]:
nltk.download("stopwords", quiet=True)
stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()

def preprocess_text(text):
    text = str(text)
    # Conversion en minuscule
    text = text.lower()
    # Suppression de la ponctuation et des caractères spéciaux
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    # Suppression des espaces multiples
    text = re.sub(r"\s+", " ", text).strip()
    # Tokenisation
    tokens = text.split()
    # Suppression des mots vides
    tokens = [word for word in tokens if word not in stop_words]
    # Racination
    tokens = [stemmer.stem(word) for word in tokens]

    return " ".join(tokens)

df_text["preprocessed_text"] = df_text["REVIEW_TEXT"].apply(preprocess_text)
df_display = df_text[["REVIEW_TEXT", "preprocessed_text"]].head()
df_display.map(lambda value: str(value)[:60] + "...")

,REVIEW_TEXT,preprocessed_text
0,"When least you think so, this product will save the day. Jus...",least think product save day keep around case need someth...
1,Lithium batteries are something new introduced in the market...,lithium batteri someth new introduc market averag develop co...
2,I purchased this swing for my baby. She is 6 months now and ...,purchas swing babi 6 month pretti much grown loud swing well...
3,I was looking for an inexpensive desk calcolatur and here it...,look inexpens desk calcolatur work everyth need issu tilt sl...
4,I only use it twice a week and the results are great. I have...,use twice week result great use teeth whiten solut result wo...


In [12]:
df_text[
    [
        "DOC_ID",
        "REVIEW_TEXT",
        "preprocessed_text",
        "VERIFIED_PURCHASE",
        "review_word_count",
        "LABEL",
        "label_num"
    ]
].head()

,DOC_ID,REVIEW_TEXT,preprocessed_text,VERIFIED_PURCHASE,review_word_count,LABEL,label_num
0,1,"When least you think so, this product will save the day. Just keep it around just in case you need it for something.",least think product save day keep around case need someth,N,23,fake,1
1,2,Lithium batteries are something new introduced in the market there average developing cost is relatively high but Stallion doesn't compromise on quality and provides us with the best at a low cost...,lithium batteri someth new introduc market averag develop cost rel high stallion compromis qualiti provid us best low cost br mani built technic assist act like sensor particular fort batteri keep...,Y,69,fake,1
2,3,I purchased this swing for my baby. She is 6 months now and has pretty much out grown it. It is very loud and doesn't swing very well. It is beautiful though. I love the colors and it has a lot of...,purchas swing babi 6 month pretti much grown loud swing well beauti though love color lot set think worth money,N,52,fake,1
3,4,I was looking for an inexpensive desk calcolatur and here it is. It works and does everything I need. Only issue is that it tilts slightly to one side so when I hit any keys it rocks a little bit....,look inexpens desk calcolatur work everyth need issu tilt slightli one side hit key rock littl bit big deal,N,44,fake,1
4,5,"I only use it twice a week and the results are great. I have used other teeth whitening solutions and most of them, for the same results I would have to use it at least three times a week. Will ke...",use twice week result great use teeth whiten solut result would use least three time week keep use potenc solut also techniqu tray keep everyth teeth mouth,N,66,fake,1


## Vérification des textes après prétraitement

Après le prétraitement, nous vérifions si certains avis sont devenus vides.

In [ ]:
empty_texts = df_text[df_text["preprocessed_text"].str.strip() == ""]

print("Nombre de textes vides :", empty_texts.shape[0])

Nombre de textes vides : 0


## Préparation du dataset final

Nous préparons un dataset final contenant les informations nécessaires pour les prochaines étapes.  
Nous conservons l’identifiant de l’avis, le texte original, le texte prétraité, le statut d’achat vérifié, la longueur initiale de l’avis, ainsi que les labels textuel et numérique.

In [ ]:
df_preprocessed = df_text[
    [
        "DOC_ID",
        "REVIEW_TEXT",
        "preprocessed_text",
        "VERIFIED_PURCHASE",
        "review_word_count",
        "LABEL",
        "label_num"
    ]
].copy()

df_preprocessed.head()

,DOC_ID,REVIEW_TEXT,preprocessed_text,VERIFIED_PURCHASE,review_word_count,LABEL,label_num
0,1,"When least you think so, this product will save the day. Just keep it around just in case you need it for something.",least think product save day keep around case need someth,N,23,fake,1
1,2,Lithium batteries are something new introduced in the market there average developing cost is relatively high but Stallion doesn't compromise on quality and provides us with the best at a low cost...,lithium batteri someth new introduc market averag develop cost rel high stallion compromis qualiti provid us best low cost br mani built technic assist act like sensor particular fort batteri keep...,Y,69,fake,1
2,3,I purchased this swing for my baby. She is 6 months now and has pretty much out grown it. It is very loud and doesn't swing very well. It is beautiful though. I love the colors and it has a lot of...,purchas swing babi 6 month pretti much grown loud swing well beauti though love color lot set think worth money,N,52,fake,1
3,4,I was looking for an inexpensive desk calcolatur and here it is. It works and does everything I need. Only issue is that it tilts slightly to one side so when I hit any keys it rocks a little bit....,look inexpens desk calcolatur work everyth need issu tilt slightli one side hit key rock littl bit big deal,N,44,fake,1
4,5,"I only use it twice a week and the results are great. I have used other teeth whitening solutions and most of them, for the same results I would have to use it at least three times a week. Will ke...",use twice week result great use teeth whiten solut result would use least three time week keep use potenc solut also techniqu tray keep everyth teeth mouth,N,66,fake,1


In [ ]:
print("Nombre de lignes :", df_preprocessed.shape[0])
print("Nombre de colonnes :", df_preprocessed.shape[1])

Nombre de lignes : 21000
Nombre de colonnes : 7


In [ ]:
df_preprocessed.isnull().sum()

DOC_ID               0
REVIEW_TEXT          0
preprocessed_text    0
VERIFIED_PURCHASE    0
review_word_count    0
LABEL                0
label_num            0
dtype: int64

Le dataset final contient 21 000 avis et 7 colonnes.  
Aucune valeur manquante n’est présente dans les colonnes conservées.

## Sauvegarde du dataset prétraité

Nous sauvegardons le dataset prétraité afin de pouvoir l’utiliser directement dans les prochaines étapes, notamment la représentation du texte avec Bag of Words et TF-IDF.

In [ ]:
output_path = "../data/amazon_reviews_preprocessed.csv"

df_preprocessed.to_csv(output_path, index=False)

print("Dataset prétraité sauvegardé dans :", output_path)

Dataset prétraité sauvegardé dans : ../data/amazon_reviews_preprocessed.csv


In [ ]:
df_check = pd.read_csv(output_path)

df_check.head()

,DOC_ID,REVIEW_TEXT,preprocessed_text,VERIFIED_PURCHASE,review_word_count,LABEL,label_num
0,1,"When least you think so, this product will save the day. Just keep it around just in case you need it for something.",least think product save day keep around case need someth,N,23,fake,1
1,2,Lithium batteries are something new introduced in the market there average developing cost is relatively high but Stallion doesn't compromise on quality and provides us with the best at a low cost...,lithium batteri someth new introduc market averag develop cost rel high stallion compromis qualiti provid us best low cost br mani built technic assist act like sensor particular fort batteri keep...,Y,69,fake,1
2,3,I purchased this swing for my baby. She is 6 months now and has pretty much out grown it. It is very loud and doesn't swing very well. It is beautiful though. I love the colors and it has a lot of...,purchas swing babi 6 month pretti much grown loud swing well beauti though love color lot set think worth money,N,52,fake,1
3,4,I was looking for an inexpensive desk calcolatur and here it is. It works and does everything I need. Only issue is that it tilts slightly to one side so when I hit any keys it rocks a little bit....,look inexpens desk calcolatur work everyth need issu tilt slightli one side hit key rock littl bit big deal,N,44,fake,1
4,5,"I only use it twice a week and the results are great. I have used other teeth whitening solutions and most of them, for the same results I would have to use it at least three times a week. Will ke...",use twice week result great use teeth whiten solut result would use least three time week keep use potenc solut also techniqu tray keep everyth teeth mouth,N,66,fake,1


Le fichier prétraité a été sauvegardé avec succès au format CSV.

Ce fichier pourra être utilisé directement dans les prochaines étapes, notamment pour la représentation du texte avec Bag of Words et TF-IDF.